# AdaptiveDepth Phase 8.1 runtime-signal pilot
This is a bounded go/no-go experiment, not another open-ended phase. It collects four-token logit confidence and a shared early hidden state for Route 20 and Route 23 on the existing 400 prompts. Continue only if grouped evaluation reaches the locked safety/coverage/latency pilot gate. Use a T4 GPU.

In [ ]:
import base64, os, pathlib, shutil, subprocess, sys, zipfile
REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
repo = pathlib.Path('/content/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs')
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
env = os.environ.copy()
token = secret('GITHUB_TOKEN')
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'Authorization: Basic {credential}'})
if not repo.exists():
    result = subprocess.run(['git','clone',REPO_URL,str(repo)],env=env,capture_output=True,text=True)
else:
    result = subprocess.run(['git','pull','--ff-only','origin','main'],cwd=repo,env=env,capture_output=True,text=True)
if result.returncode:
    raise RuntimeError(result.stderr.strip() or 'Repository synchronization failed')
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-e','.[router]'],check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('Commit:',subprocess.run(['git','rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_results = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
checkpoint_archive = drive_results / 'phase5_corrected_student.zip'
phase61_archive = drive_results / 'phase61_artifacts.zip'
judgments = drive_results / 'phase72_label_judgments.jsonl'
for required in (checkpoint_archive,phase61_archive,judgments):
    if not required.is_file():
        raise FileNotFoundError(f'Missing required Drive file: {required}')
checkpoint_root = pathlib.Path('/content/phase5_for_phase81')
phase61_root = pathlib.Path('/content/phase61_for_phase81')
for target in (checkpoint_root,phase61_root):
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
with zipfile.ZipFile(checkpoint_archive) as bundle:
    bundle.extractall(checkpoint_root)
with zipfile.ZipFile(phase61_archive) as bundle:
    bundle.extractall(phase61_root)
checkpoint_matches = [path.parent for path in checkpoint_root.rglob('model.safetensors')]
dataset_matches = list(phase61_root.rglob('router_dataset.jsonl'))
if len(checkpoint_matches) != 1 or len(dataset_matches) != 1:
    raise RuntimeError(f'Unexpected checkpoint/dataset matches: {checkpoint_matches}, {dataset_matches}')
checkpoint = checkpoint_matches[0]
original_dataset = dataset_matches[0]
application_dataset = pathlib.Path('/content/phase72_application_dataset.jsonl')
subprocess.run([
    'adaptivedepth','apply-phase72-label-review',
    '--dataset',str(original_dataset),'--judgments',str(judgments),
    '--output',str(application_dataset),'--minimum-safe-confidence','0.80',
],check=True)
print('Checkpoint:',checkpoint)
print('Corrected prompts:',sum(1 for line in application_dataset.open(encoding='utf-8') if line.strip()))

In [ ]:
MAX_EXAMPLES = 400
probe_output = drive_results / 'phase81_probe_signals.jsonl'
command = [
    'adaptivedepth','collect-phase81-probes',
    '--checkpoint',str(checkpoint),'--dataset',str(application_dataset),
    '--config','configs/phase81_probe_t4.yaml',
    '--routes-file','configs/phase61_routes.yaml',
    '--output',str(probe_output),'--max-examples',str(MAX_EXAMPLES),'--resume',
]
print('Expected route rows:',MAX_EXAMPLES * 2)
print('Resumable output:',probe_output)

In [ ]:
process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='')
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Probe collection failed with exit code {return_code}; see output above')

In [ ]:
import json, collections
rows = [json.loads(line) for line in probe_output.read_text(encoding='utf-8').splitlines() if line.strip()]
keys = {(row['example_id'],row['route_id']) for row in rows}
print('Rows:',len(rows),'unique:',len(keys),'expected:',MAX_EXAMPLES * 2)
print('Routes:',collections.Counter(row['route_id'] for row in rows))
print('Saved:',probe_output)
print('Download this JSONL or add it to the project folder for the CPU-only go/no-go evaluation.')